# Lab 03: Complex SQL Joins, Views, CTEs & Window Functions

**Z2004 Database Management Systems (BS Core) · IIT Madras Zanzibar**

Multi-table LEFT JOIN, SQL Views, Common Table Expressions, and RANK() window analytics.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z2004/labs/DB03_SQL_Joins_CTEs_Window_Functions.ipynb)

Run cells top to bottom (`Runtime > Run all`). Built on Python `sqlite3` with full self-checks.

## 1. Joins, Views, CTEs & Window Functions on Campus Canteen Orders

Using `LEFT JOIN` with `COALESCE` retains zero-order students (`Juma`), while `RANK() OVER (PARTITION BY Hostel ORDER BY TotalSpend DESC)` computes per-hostel rankings without collapsing rows.

In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE Students (StudentID TEXT PRIMARY KEY, Name TEXT, Year INT, Hostel TEXT);
CREATE TABLE FoodItems (ItemID TEXT PRIMARY KEY, Name TEXT UNIQUE, Category TEXT, Price REAL);
CREATE TABLE Orders (OrderID INT PRIMARY KEY, StudentID TEXT, ItemName TEXT, Qty INT);

INSERT INTO Students VALUES ('S1','Asha',2,'Kizingo'),('S2','Omar',2,'Kizingo'),('S3','Zahra',3,'Mbweni'),('S4','Juma',1,'Mbweni');
INSERT INTO FoodItems VALUES ('F1','Pilau','Main',5.50),('F2','Urojo','Snack',2.00),('F3','Chai','Drink',1.00);
INSERT INTO Orders VALUES (1,'S1','Pilau',2),(2,'S1','Chai',3),(3,'S2','Urojo',4),(4,'S3','Pilau',1);

CREATE VIEW StudentSpend AS
SELECT s.StudentID, s.Name, s.Hostel, COALESCE(SUM(o.Qty * f.Price), 0.0) AS TotalSpend
FROM Students s
LEFT JOIN Orders o ON s.StudentID = o.StudentID
LEFT JOIN FoodItems f ON o.ItemName = f.Name
GROUP BY s.StudentID, s.Name, s.Hostel;
""")

rows = conn.execute("""
WITH S AS (SELECT * FROM StudentSpend)
SELECT Name, Hostel, TotalSpend,
       RANK() OVER (PARTITION BY Hostel ORDER BY TotalSpend DESC) AS rnk
FROM S ORDER BY Hostel, rnk
""").fetchall()
for r in rows:
    print(r)
assert rows[0][0] == "Asha" and rows[-1][2] == 0.0


('Asha', 'Kizingo', 14.0, 1)
('Omar', 'Kizingo', 8.0, 2)
('Zahra', 'Mbweni', 5.5, 1)
('Juma', 'Mbweni', 0.0, 2)
